# Pill classification system — Kaggle T4x2 training notebook

## What this notebook runs (in order)

| Phase | Script | Description |
|---|---|---|
| T1 | `train_dino_ssl.py` | DINO domain adaptation — **9-channel** fused pairs |
| T2+T3 | `train_tower.py` (×3 towers ×3 phases ×N folds) | LOOCV tower training |
| T4 | `train_prototype.py` | Prototypical network — 50k episodes |
| — | `build_profiles_and_ood.py` | Constraint profiles + OOD index from **ROI crops** |
| — | `generate_calibration_signals.py` | LOOCV inference pass → signals.jsonl |
| — | `build_calibrator.py` | Isotonic calibration on real signals |
| — | `export_runtime_pack.py` | Final deployable bundle |

## Dataset folder structure expected

```
your_dataset/
  DrugName_Dosage_A/              ← one folder per class
    pair_001/
      side_a.png
      side_b.png
    pair_002/
      ...
  DrugName_Dosage_B/
    ...
```

No background_reference.png required. The system uses a pure black reference automatically — correct for black-tray pharmacy setups.

Pair grouping: images are grouped into pairs by subfolder name or by filename prefix before the first underscore. Each pair must have exactly 2 images — one per side.

## Bug fixes applied vs original code
- **Bug 1**: Argument mismatch in launcher fixed; build_profiles uses black reference directly
- **Bug 2**: Constraint profiles built from preprocessed ROI crops (not raw frames)
- **Bug 3**: OOD index built from preprocessed ROI features (not raw frame features)
- **Bug 4**: `generate_calibration_signals.py` generates actual signal records before calibration
- **Issue 5**: DINO trains `DINOAdaptBackbone` (9-channel) — adapted weights load cleanly into all tower stems
- **Issue 6**: `LightROIPreprocessor` runs on training images so train/inference distributions match
- **background_reference removed**: black reference (np.zeros) used everywhere, no file needed


In [ ]:
# Cell 1 — Install dependencies and clear pip cache
import subprocess, shutil

%cd /kaggle/working

# Install required packages
subprocess.run(['pip', 'install', '-q', 'timm', 'faiss-cpu', 'scikit-learn',
                'scikit-image', 'scipy', 'joblib', 'tqdm'], check=True)

# IMPORTANT: Clear pip cache immediately after install.
# pip keeps downloaded wheels in ~/.cache/pip which wastes 1-3 GB.
subprocess.run(['pip', 'cache', 'purge'], check=False)

# Show available disk space
stat = shutil.disk_usage('/kaggle/working')
free_gb = stat.free / (1024**3)
total_gb = stat.total / (1024**3)
used_gb = stat.used / (1024**3)
print(f'Disk: {used_gb:.1f} GB used / {total_gb:.1f} GB total / {free_gb:.1f} GB free')
print('Setup complete')


In [ ]:
# Cell 2 — Extract code package and configure paths
import subprocess, os

# ── CONFIGURE THESE TWO PATHS ─────────────────────────────────────────────
CODE_ZIP  = '/kaggle/input/pill-prod-code/pill_prod_code_v3.zip'
DATA_ROOT = '/kaggle/input/your-pill-dataset'        # ← your dataset folder
WORKDIR   = '/kaggle/working/pill_run'
# ──────────────────────────────────────────────────────────────────────────

subprocess.run(['unzip', '-q', '-o', CODE_ZIP, '-d', '/kaggle/working/pill_system'], check=True)
%cd /kaggle/working/pill_system/work_exact/trainer

# Verify dataset
classes = [d for d in os.listdir(DATA_ROOT) if os.path.isdir(f'{DATA_ROOT}/{d}')]
print(f'Dataset: {DATA_ROOT}')
print(f'Classes found ({len(classes)}): {classes}')

# Quick pair count
total_pairs = 0
for cls in classes:
    cls_path = f'{DATA_ROOT}/{cls}'
    subfolders = [d for d in os.listdir(cls_path) if os.path.isdir(f'{cls_path}/{d}')]
    if subfolders:
        total_pairs += len(subfolders)
    else:
        # Files directly in class folder — count by pairs
        files = [f for f in os.listdir(cls_path) if not f.startswith('.')]
        total_pairs += len(files) // 2

print(f'Estimated total pairs: {total_pairs}')
print(f'Estimated LOOCV folds: ~{max(len(os.listdir(f"{DATA_ROOT}/{c}")) for c in classes if os.path.isdir(f"{DATA_ROOT}/{c}"))}')
print()

# Disk estimate — LOOCV folds save NO checkpoints (fixed in v3)
# Approximate disk usage:
#   DINO checkpoint:      ~300 MB
#   3 × final tower ckpt: ~870 MB
#   Prototype checkpoint: ~120 MB
#   OOD/profiles/bundle:  ~200 MB
#   pip cache (cleared):    ~0 MB
#   Total estimate:       ~1.5 GB  (well within 20 GB limit)
print('Expected peak disk usage: ~1.5-2 GB (LOOCV folds save no weights)')


In [ ]:
# ── RECOVERY CELL ─────────────────────────────────────────────────────────────
# Use this ONLY if the training run crashed at the cleanup/export step
# but the training itself completed (you saw towers + prototype finish).
#
# Check if training artifacts exist:
import os
from pathlib import Path

WORKDIR = '/kaggle/working/aimed'    # ← set to your WORKDIR
TRAINER = '/kaggle/working/aimed/trainer'

tower_a_ckpt = f'{WORKDIR}/tower_a/final/best.ckpt'
tower_b_ckpt = f'{WORKDIR}/tower_b/final/best.ckpt'
tower_c_ckpt = f'{WORKDIR}/tower_c/final/best.ckpt'
proto_ckpt   = f'{WORKDIR}/prototype/prototype_best.ckpt'

missing = []
for p in [tower_a_ckpt, tower_b_ckpt, tower_c_ckpt, proto_ckpt]:
    if os.path.exists(p):
        size_mb = os.path.getsize(p) / (1024**2)
        print(f'  OK  {p}  ({size_mb:.0f} MB)')
    else:
        missing.append(p)
        print(f'  MISSING  {p}')

if missing:
    print('\nSome artifacts are missing. Training may not have completed.')
else:
    print('\nAll training artifacts found. Running export-only to create bundle...')
    import subprocess, sys

    os.chdir(TRAINER)

    # Step 1: Build profiles and OOD index (if not already done)
    if not os.path.exists(f'{WORKDIR}/profiles.json'):
        subprocess.run(['python', 'build_profiles_and_ood.py',
            '--data-root', DATA_ROOT,
            '--output-profiles', f'{WORKDIR}/profiles.json',
            '--output-ood-prefix', f'{WORKDIR}/ood_index'], check=True)

    # Step 2: Export pre-bundle (needed for calibration signal generation)
    import joblib, numpy as np
    placeholder_cal = {
        'kind': 'multivariate_isotonic',
        'train_features': np.zeros((1, 12), dtype=np.float32),
        'fitted_values': np.asarray([0.5], dtype=np.float32),
        'feature_mins': np.zeros(12, dtype=np.float32),
        'feature_maxs': np.ones(12, dtype=np.float32),
        'eps': 1e-9,
    }
    cal_path = f'{WORKDIR}/calibrator_placeholder.joblib'
    joblib.dump(placeholder_cal, cal_path)
    var_path = f'{WORKDIR}/variance_thresholds_placeholder.json'
    open(var_path, 'w').write('{}')
    labels_path = f'{WORKDIR}/tower_a/final/labels.json'
    pre_export = f'{WORKDIR}/runtime_bundle_pre'

    subprocess.run(['python', 'export_runtime_pack.py',
        '--tower-a',              tower_a_ckpt,
        '--tower-b',              tower_b_ckpt,
        '--tower-c',              tower_c_ckpt,
        '--prototype-checkpoint', proto_ckpt,
        '--prototype-library',    f'{WORKDIR}/prototype/prototype_library.json',
        '--labels',               labels_path,
        '--class-profiles',       f'{WORKDIR}/profiles.json',
        '--variance-thresholds',  var_path,
        '--ood-index-prefix',     f'{WORKDIR}/ood_index',
        '--calibrator',           cal_path,
        '--output-dir',           pre_export], check=True)

    # Step 3: Sync pre-bundle and generate calibration signals
    import shutil
    runtime_current = Path(TRAINER).parent / 'runtime_service' / 'models' / 'current'
    runtime_current.mkdir(parents=True, exist_ok=True)
    for f in Path(pre_export).iterdir():
        shutil.copy2(f, runtime_current / f.name)

    from configs_for_calibration import write_calibration_config
    cal_config = f'{WORKDIR}/cal_config.yaml'
    write_calibration_config(str(runtime_current), cal_config, labels_path=labels_path)

    signals_path = f'{WORKDIR}/signals.jsonl'
    subprocess.run(['python', 'generate_calibration_signals.py',
        '--data-root',     DATA_ROOT,
        '--config-path',   cal_config,
        '--output-signals', signals_path], check=True)

    # Step 4: Calibrate
    subprocess.run(['python', 'build_calibrator.py',
        '--signals-jsonl',              signals_path,
        '--output',                     f'{WORKDIR}/calibrator.joblib',
        '--variance-thresholds-output', f'{WORKDIR}/variance_thresholds.json',
        '--reliability-output',         f'{WORKDIR}/reliability.json'], check=True)

    # Step 5: Final export
    subprocess.run(['python', 'export_runtime_pack.py',
        '--tower-a',              tower_a_ckpt,
        '--tower-b',              tower_b_ckpt,
        '--tower-c',              tower_c_ckpt,
        '--prototype-checkpoint', proto_ckpt,
        '--prototype-library',    f'{WORKDIR}/prototype/prototype_library.json',
        '--labels',               labels_path,
        '--class-profiles',       f'{WORKDIR}/profiles.json',
        '--variance-thresholds',  f'{WORKDIR}/variance_thresholds.json',
        '--ood-index-prefix',     f'{WORKDIR}/ood_index',
        '--calibrator',           f'{WORKDIR}/calibrator.joblib',
        '--output-dir',           f'{WORKDIR}/runtime_bundle'], check=True)

    print('\nRecovery complete! Bundle at:', f'{WORKDIR}/runtime_bundle')


In [ ]:
# Cell 3 — Run full training pipeline (expected: 4-8 hours on T4x2)
import subprocess
result = subprocess.run([
    'python', 'notebook_launcher.py',
    '--data-root', DATA_ROOT,
    '--workdir',   WORKDIR,
], check=False)

if result.returncode != 0:
    print(f'\nTraining failed with exit code {result.returncode}')
    print('Check the output above for the error message.')
else:
    print('\nTraining complete!')


In [ ]:
# Cell 4 — Verify the output bundle
import os, json, shutil, joblib, numpy as np

BUNDLE = f'{WORKDIR}/runtime_bundle'

# Check disk
stat = shutil.disk_usage('/kaggle/working')
free_gb = stat.free / (1024**3)
used_gb = stat.used / (1024**3)
print(f'Disk used: {used_gb:.1f} GB  free: {free_gb:.1f} GB')

manifest_path = f'{BUNDLE}/pack_manifest.json'
assert os.path.exists(manifest_path), 'pack_manifest.json missing — training may have failed'

manifest = json.loads(open(manifest_path).read())
print('\n=== Runtime bundle contents ===')
for name, info in manifest['files'].items():
    size_kb = info['bytes'] / 1024
    print(f'  {name:<40}  {size_kb:>10.1f} KB')

# Verify calibrator is not a placeholder
cal = joblib.load(f'{BUNDLE}/calibrator.joblib')
n_cal = len(cal.get('train_features', []))
print(f'\nCalibrator trained on {n_cal} signal records')
assert n_cal > 1, 'Calibrator is a placeholder — signal generation failed'

# Show LOOCV summary
summary_path = f'{WORKDIR}/loocv_summary.json'
if os.path.exists(summary_path):
    summary = json.loads(open(summary_path).read())
    from collections import defaultdict
    tower_phase = defaultdict(list)
    for row in summary:
        tower_phase[(row['tower'], row['phase'])].append(row['best_acc'])
    print('\n=== LOOCV accuracy per tower/phase ===')
    for (tower, phase), accs in sorted(tower_phase.items()):
        print(f'  {tower} {phase}: mean={sum(accs)/len(accs):.3f}  min={min(accs):.3f}  max={max(accs):.3f}')

print('\nBundle verified successfully.')


In [ ]:
# Cell 5 — Zip the bundle for download
import shutil, os

OUTPUT_ZIP = '/kaggle/working/runtime_bundle.zip'
shutil.make_archive(
    OUTPUT_ZIP.replace('.zip', ''),
    'zip',
    f'{WORKDIR}/runtime_bundle'
)
size_mb = os.path.getsize(OUTPUT_ZIP) / (1024**2)
print(f'Bundle zipped: {OUTPUT_ZIP}  ({size_mb:.1f} MB)')
print('Download it from the Output tab on the right side of the Kaggle notebook page.')
